# Lab 06: Collect published statistical data from a website (solution)

You will check permission, download a real web page, find the exports table, extract and clean its rows, and record where the data came from.

**Source:** Wikipedia, *List of countries by exports* (text CC BY-SA 4.0; figures from the World Bank). No internet? The cached copy `data/cache/wikipedia_exports.html` is used automatically from Part B onwards; skip A2 and A3.

---
# Part A: Are we allowed?

## A1. Set up
Imports, the page address and a polite User-Agent header that identifies us.

*Run this cell; no changes needed.*

In [ ]:
import re
from datetime import date
from io import StringIO
from pathlib import Path
from urllib import robotparser

import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/wiki/List_of_countries_by_exports"
HEADERS = {"User-Agent": "TradeCourse/1.0 (training exercise)"}
CACHE = Path("../../data/cache")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

## A2. Download robots.txt
robots.txt tells automated clients what they may fetch. Download `https://en.wikipedia.org/robots.txt` with `requests.get`, passing `headers=HEADERS` and `timeout=30`.

In [ ]:
robots = requests.get("https://en.wikipedia.org/robots.txt", headers=HEADERS, timeout=30)
print(robots.status_code)
print(robots.text[:300])

In [ ]:
# check
assert robots.status_code == 200
print("A2 OK")

## A3. Ask whether our URL is allowed
`RobotFileParser` reads the rules; `can_fetch(agent, url)` answers yes or no. Complete the two lines.

(Why not `rp.read()`? It downloads robots.txt with a default user agent that Wikipedia blocks, so it wrongly says *not allowed*.)

In [ ]:
rp = robotparser.RobotFileParser()
rp.parse(robots.text.splitlines())
allowed = rp.can_fetch(HEADERS["User-Agent"], URL)
print(allowed)

In [ ]:
# check
assert allowed is True
print("A3 OK")

**Question:** Apart from robots.txt, name two things to check before scraping a site.

*Your answer:* The site's terms of use and the content licence (Wikipedia: CC BY-SA, attribution required); whether an API or download exists instead; keeping request volume low; whether the page contains personal data.

---
# Part B: Download the page

## B1. Download with a fallback
Write `get_html(url)`: **try** to download the page (headers, timeout, `raise_for_status()`) and return `resp.text`. If a `requests.RequestException` occurs, print a message and return the cached file's text instead.

In [ ]:
def get_html(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        return resp.text
    except requests.RequestException as e:
        print("Download failed, using cache:", e)
        return (CACHE / "wikipedia_exports.html").read_text(encoding="utf-8")

html = get_html(URL)
print(len(html), "characters")

In [ ]:
# check
assert "<table" in html
print("B1 OK")

---
# Part C: Find the table

## C1. Parse the HTML
BeautifulSoup turns the HTML text into a tree we can search.

*Run this cell; no changes needed.*

In [ ]:
soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text())

## C2. How many tables?
`soup.find_all("table")` returns every table. Store them in `tables` and print how many there are.

In [ ]:
tables = soup.find_all("table")
print(len(tables))

In [ ]:
# check
assert len(tables) >= 1
print("C2 OK")

## C3. Pick the right table by its caption
Loop over `tables`. For each, `t.find("caption")` returns the caption (or `None`). When the caption text contains `"Exports"`, store the table in `table` and `break` out of the loop.

In [ ]:
table = None
for t in tables:
    cap = t.find("caption")
    if cap is not None and "Exports" in cap.get_text():
        table = t
        break

print(table.find("caption").get_text(strip=True))

In [ ]:
# check
assert table is not None
print("C3 OK")

## C4. The header row
`table.find_all("tr")` gives the rows. The first row holds the headers in `<th>` cells. Build `headers`: the text of each `<th>` in the first row.

In [ ]:
trs = table.find_all("tr")
headers = []
for th in trs[0].find_all("th"):
    headers.append(th.get_text(" ", strip=True))
print(headers)

In [ ]:
# check
assert len(headers) == 4
print("C4 OK")

## C5. The data rows
For every row after the first (`trs[1:]`), collect the text of its cells (`tr.find_all(["th", "td"])`) into a list, and append that list to `rows`.

In [ ]:
rows = []
for tr in trs[1:]:
    cells = []
    for c in tr.find_all(["th", "td"]):
        cells.append(c.get_text(" ", strip=True))
    rows.append(cells)
print(len(rows))
print(rows[0])

In [ ]:
# check
assert len(rows) > 150 and len(rows[0]) == 4
print("C5 OK")

---
# Part D: Clean the values

## D1. Remove footnote markers
Values can contain footnotes such as `2025 [3]`. `re.sub(r"\[.*?\]", "", text)` deletes anything in square brackets. Write `clean_text(text)` that does this and strips spaces.

In [ ]:
def clean_text(text):
    return re.sub(r"\[.*?\]", "", text).strip()

print(repr(clean_text("2025 [3]")))

In [ ]:
# check
assert clean_text("2025 [3]") == "2025" and clean_text(" Kenya ") == "Kenya"
print("D1 OK")

## D2. Build a DataFrame
Create `exports` from `rows`, applying `clean_text` to every cell, with columns `country`, `exports_usd_m`, `year`, `top_export`.

In [ ]:
clean_rows = []
for r in rows:
    clean_rows.append([clean_text(c) for c in r])
exports = pd.DataFrame(clean_rows, columns=["country", "exports_usd_m", "year", "top_export"])
exports.head()

In [ ]:
# check
assert len(exports) == len(rows)
assert not exports["year"].str.contains(r"\[").any()
print("D2 OK")

## D3. Convert the types
Scraped values are text. Remove the commas from `exports_usd_m` and convert to float; convert `year` to int.

**Example**
```python
df["x"] = df["x"].str.replace(",", "").astype(float)
```

In [ ]:
exports["exports_usd_m"] = exports["exports_usd_m"].str.replace(",", "").astype(float)
exports["year"] = exports["year"].astype(int)
exports.dtypes

In [ ]:
# check
assert exports["exports_usd_m"].dtype == float and exports["year"].dtype.kind == "i"
print("D3 OK")

## D4. Record where it came from
Add a `source` column holding `URL`, and a `retrieved` column holding today's date (`date.today().isoformat()`).

In [ ]:
exports["source"] = URL
exports["retrieved"] = date.today().isoformat()
exports.head(2)

In [ ]:
# check
assert {"source", "retrieved"} <= set(exports.columns)
print("D4 OK")

---
# Part E: Compare and check quality

## E1. The shortcut: pandas.read_html
`pd.read_html` reads every matching table in one call. Compare the result with yours: what would you still need to clean?

*Run this cell; no changes needed.*

In [ ]:
rh = pd.read_html(StringIO(html), match="Exports")[0]
print(len(rh), "rows via read_html vs", len(exports), "rows via BeautifulSoup")
rh.head()

## E2. Which years?
Show how many countries have data for each year: `exports["year"].value_counts().sort_index()`.

In [ ]:
exports["year"].value_counts().sort_index()

**Question:** Why does it matter that countries report different years when you rank them?

*Your answer:* A ranking may compare 2021 figures for one country with 2024 figures for another, mixing growth, price changes and different reference periods.

## E3. Do our country names match?
Load `countries.xlsx` and find the course economies whose names are **not** in the scraped table. Hint: `set(a) - set(b)` gives items in `a` but not in `b`.

In [ ]:
countries = pd.read_excel("../../data/countries.xlsx")
missing_names = set(countries["country_name"]) - set(exports["country"])
print(missing_names)

In [ ]:
# check
assert "Viet Nam" in missing_names
print("E3 OK: Wikipedia uses 'Vietnam'. Module 07 shows how to fix this")

## E4. Save
Save `exports` to `output/web_exports.csv` without the index.

In [ ]:
exports.to_csv(OUT / "web_exports.csv", index=False)

In [ ]:
# check
assert (OUT / "web_exports.csv").exists()
print("E4 OK")